# 📈 네이버 검색 트렌드 데이터 수집
## 1. 주요 기능
- **배치 비교 조회 및 정량화 (Anchor Calibration)**: 네이버 API의 하루 한도(1,000회) 내에서 최대한 많은 영화를 효율적으로 수집하기 위해, **영화 4편과 상시 고정 기준 키워드(Anchor Keyword) 1개를 1개 배치(총 5개 그룹)**로 묶어 호출합니다.
- **기준점 정규화를 통한 절대 스케일 통일**: 검색량이 연중 높고 매우 일정한 공통 기준 키워드(Anchor Keyword: `"날씨"`)를 배치마다 1번 자리에 고정 삽입해 호출함으로써, 서로 다른 API 세션에서 수집된 모든 영화의 검색 지수 단위를 하나의 일관된 절대적 척도로 보정합니다.
- **동적 상대 기간 산정**: 각 영화의 개봉일 기준 **D-30 ~ D-1** 기간의 데이터를 조회하기 위해, 배치 내 영화들의 개봉일을 동적으로 반영한 공통 조회 기간을 설정합니다.
- **증분 수집 (Delta Load)**: 이미 `naver_search_trend` 테이블에 적재되어 있는 영화는 수집 대상에서 제외하여, 중복 API 호출을 방지합니다.
- **API 조회 하한선 고려**: 네이버 검색 트렌드는 **2016년 1월 1일** 데이터부터 제공합니다. 개봉 전 한 달(30일) 데이터의 완전성을 유지하기 위해 **개봉일이 2016년 2월 1일 이후인 영화만 수집**합니다.

In [1]:
from datetime import date, timedelta
import time
import requests
from data.api import get_naver_search_trend
from data.api.naver_dto import KeywordGroupDTO
from data.db import db

movie_open_start_date = '2016-02-01'

### 1. 수집 대상 영화 로드 및 중복 필터링 (Delta Load)
- `movies` 테이블에서 네이버 API 수집이 가능한 **2016-02-01 이후 개봉작**을 가져옵니다. (개봉 전 D-30 조회 범위가 2016-01-01 이후가 되도록 보장)
- 이미 `naver_search_trend` 테이블에 1건이라도 데이터가 쌓인 영화는 수집 완료로 간주해 제외합니다.

In [2]:
def get_collection_targets():
    """
    수집이 필요한 영화 목록을 데이터베이스에서 조회하고,
    이미 수집된 대상을 제외한 미수집 대상 리스트를 반환합니다.
    """
    # 1. 2016년 2월 1일 이후 개봉 영화 목록 조회
    # 개봉 전 D-30일 트렌드 데이터를 네이버 데이터랩 API 한계(2016-01-01부터 조회 가능) 내에서 온전히 가져오기 위함
    movies = db.fetch_all("""
        SELECT movie_id, title, open_date
        FROM movies
        WHERE open_date >= %s
    """, (movie_open_start_date,))
    
    # 2. 이미 데이터가 적재된 영화 ID 조회
    done = db.fetch_all("SELECT DISTINCT movie_id FROM naver_search_trend")
    done_ids = {row['movie_id'] for row in done}
    
    # 3. 미수집 대상 필터링
    targets = [m for m in movies if m['movie_id'] not in done_ids]
    print(f"🎬 전체 대상 영화({movie_open_start_date} 이후): {len(movies)}편 | 🔄 미수집 영화: {len(targets)}편")
    return targets

### 2. 배치 그룹 빌드 (5편씩 묶기)
- 검색 지수의 비교 타당성을 높이기 위해 **개봉일이 인접한 영화끼리 5편씩 정렬하여 묶습니다**.

In [3]:
def build_batches(targets, batch_size=4):
    """
    대상 영화 목록을 개봉일 순서로 정렬한 후 batch_size(최대 4) 크기의 묶음으로 분할합니다.
    """
    targets_sorted = sorted(targets, key=lambda m: m['open_date'])
    batches = [targets_sorted[i:i+batch_size] for i in range(0, len(targets_sorted), batch_size)]
    print(f"📦 총 {len(batches)}개의 수집 배치(그룹)가 생성되었습니다. (배치당 최대 {batch_size}편)")
    return batches

### 3. Bulk Insert 함수 작성
- 수집된 트렌드 원시 데이터를 DB에 안전하게 Bulk Write 합니다.
- 중복 데이터 발생 시 멱등성을 보장하기 위해 `ON DUPLICATE KEY UPDATE` 전략을 사용합니다.

In [4]:
def bulk_insert_trends(rows):
    """
    네이버 검색 트렌드 데이터 리스트를 DB에 Bulk Insert 합니다.
    """
    query = """
    INSERT INTO naver_search_trend (
        movie_id, trend_date, search_index, query_period_start, query_period_end
    ) VALUES (%s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        search_index = VALUES(search_index),
        query_period_start = VALUES(query_period_start),
        query_period_end = VALUES(query_period_end),
        updated_at = CURRENT_TIMESTAMP
    """
    affected = db.execute_many(query, rows)
    return affected

### 4. 전체 수집 파이프라인 실행 로직
- 각 배치 내부에서 영화들의 개봉일을 조회하여 공통 조회 기간을 D-30 ~ D-1 기반으로 계산합니다.
- 네이버 API를 호출하고 응답 DTO를 DB 튜플 구조로 파싱해 Bulk 적재를 수행합니다.
- **치명적 에러 방어**: API 호출 한도(429)나 인증 에러(401, 403) 발생 시, 프로세스를 무리하게 돌리지 않고 조기 종료하여 이전에 수집된 데이터만 안전하게 보존합니다.

In [5]:
def fetch_and_save_naver_trends(batch_limit=None):
    """
    미수집된 영화의 네이버 트렌드 수집을 총괄 실행합니다.
    
    Args:
        batch_limit (int, optional): 수집을 시도할 최대 배치 개수. 테스트 용도로 활용.
    """
    # 1. 수집 대상 로드
    targets = get_collection_targets()
    if not targets:
        print("✅ 수집 대상 영화가 존재하지 않습니다.")
        return
        
    # 2. 배치 빌드 (앵커 키워드 포함을 위해 배치 크기 4 적용)
    batches = build_batches(targets, batch_size=4)
    if batch_limit:
        batches = batches[:batch_limit]
        print(f"⚠️ [테스트 모드] 상위 {batch_limit}개 배치만 처리합니다.")
        
    total_inserted = 0
    success_batches = 0
    ANCHOR_KEYWORD = "날씨"
    
    for idx, batch in enumerate(batches):
        print(f"\n🚀 배치 [{idx + 1}/{len(batches)}] 처리 시작...")
        
        # 3. 공통 조회 기간 계산
        # D-30 ~ D-1 범위를 확보하기 위해 배치 내 최소 개봉일과 최대 개봉일을 구합니다.
        earliest_open = min(m['open_date'] for m in batch)
        latest_open = max(m['open_date'] for m in batch)
        
        start_dt = earliest_open - timedelta(days=30)
        end_dt = latest_open - timedelta(days=1)
        
        start_date_str = start_dt.strftime('%Y-%m-%d')
        end_date_str = end_dt.strftime('%Y-%m-%d')
        
        print(f"  - 조회 기간: {start_date_str} ~ {end_date_str} (개봉 분포: {earliest_open} ~ {latest_open})")
        
        # 4. keywordGroups 구성 (요청 스키마 DTO 맞춤형)
        # 스케일 정규화 기준점 보정을 위해 상시 검색량이 매우 높고 일정한 앵커 키워드(ANCHOR_KEYWORD)를 1번에 삽입합니다.
        keyword_groups = [
            KeywordGroupDTO(groupName=ANCHOR_KEYWORD, keywords=[ANCHOR_KEYWORD])
        ] + [
            KeywordGroupDTO(groupName=m['title'], keywords=[m['title']])
            for m in batch
        ]
        
        title_to_id = {m['title']: m['movie_id'] for m in batch}
        
        # 5. API 호출
        try:
            response = get_naver_search_trend(
                start_date=start_date_str,
                end_date=end_date_str,
                time_unit="date",
                keyword_groups=keyword_groups
            )
            
            if not response or not response.results:
                print(f"  ❌ 배치 [{idx + 1}] API 응답이 없거나 결과 리스트가 비어 있습니다. (건너뜀)")
                continue
                
            # 6. 결과 파싱 및 삽입 튜플 구축
            rows = []
            for res in response.results:
                if res.title == ANCHOR_KEYWORD:
                    continue  # 앵커 키워드 자체는 DB에 적재하지 않고 스케일 보정용으로만 사용합니다.
                    
                movie_id = title_to_id.get(res.title)
                if not movie_id:
                    print(f"  ⚠️ DTO 응답 내 주제어 '{res.title}'에 해당하는 영화 ID를 찾을 수 없습니다.")
                    continue
                    
                for d in res.data:
                    rows.append((
                        movie_id,
                        d.period,       # trend_date
                        d.ratio,        # search_index
                        start_date_str, # query_period_start
                        end_date_str    # query_period_end
                    ))
            
            # 7. DB 적재 실행
            if rows:
                inserted = bulk_insert_trends(rows)
                total_inserted += inserted
                success_batches += 1
                print(f"  ✅ 배치 [{idx + 1}] 저장 완료: {inserted}건의 트렌드 레코드 적재")
            else:
                print(f"  ⚠️ 배치 [{idx + 1}] 파싱할 유효 트렌드 레코드가 존재하지 않습니다.")
                
        except requests.exceptions.HTTPError as e:
            status_code = e.response.status_code if e.response is not None else None
            if status_code in (401, 403, 429):
                print(f"\n🚨 치명적 API 오류 발생 (상태 코드: {status_code}). 인증 오류이거나 일일 호출 한도(1000회)를 초과했습니다.")
                print("지금까지 수집된 데이터는 이미 DB에 안전하게 저장되었습니다. 전체 수집 파이프라인을 조기 종료합니다.")
                break
            else:
                print(f"  ❌ 배치 [{idx + 1}] 실행 중 HTTP 에러 발생: {e}")
        except Exception as e:
            print(f"  ❌ 배치 [{idx + 1}] 실행 중 에러 발생: {e}")
            
        # Rate Limit 준수를 위한 API 딜레이 타임 슬립
        time.sleep(0.35)
        
    print(f"\n🎉 수집 파이프라인 완료! 총 {success_batches}개 배치 수집 성공, 누적 {total_inserted}건의 검색량 트렌드 레코드 적재 완료.")

In [8]:
# 테스트 수집 실행 (1개 배치: 최대 5개 영화 수집 동작)
# fetch_and_save_naver_trends(batch_limit=1)

# 전체 수집 실행 (주석 해제 후 실행)
fetch_and_save_naver_trends()



🎬 전체 대상 영화(2016-02-01 이후): 2471편 | 🔄 미수집 영화: 2463편
📦 총 616개의 수집 배치(그룹)가 생성되었습니다. (배치당 최대 4편)

🚀 배치 [1/616] 처리 시작...
  - 조회 기간: 2016-01-18 ~ 2016-02-16 (개봉 분포: 2016-02-17 ~ 2016-02-17)
  ✅ 배치 [1] 저장 완료: 120건의 트렌드 레코드 적재

🚀 배치 [2/616] 처리 시작...
  - 조회 기간: 2016-01-18 ~ 2016-02-23 (개봉 분포: 2016-02-17 ~ 2016-02-24)
  ✅ 배치 [2] 저장 완료: 148건의 트렌드 레코드 적재

🚀 배치 [3/616] 처리 시작...
  - 조회 기간: 2016-01-25 ~ 2016-02-24 (개봉 분포: 2016-02-24 ~ 2016-02-25)
  ✅ 배치 [3] 저장 완료: 124건의 트렌드 레코드 적재

🚀 배치 [4/616] 처리 시작...
  - 조회 기간: 2016-02-02 ~ 2016-03-09 (개봉 분포: 2016-03-03 ~ 2016-03-10)
  ✅ 배치 [4] 저장 완료: 148건의 트렌드 레코드 적재

🚀 배치 [5/616] 처리 시작...
  - 조회 기간: 2016-02-09 ~ 2016-03-16 (개봉 분포: 2016-03-10 ~ 2016-03-17)
  ✅ 배치 [5] 저장 완료: 148건의 트렌드 레코드 적재

🚀 배치 [6/616] 처리 시작...
  - 조회 기간: 2016-02-16 ~ 2016-03-23 (개봉 분포: 2016-03-17 ~ 2016-03-24)
  ✅ 배치 [6] 저장 완료: 148건의 트렌드 레코드 적재

🚀 배치 [7/616] 처리 시작...
  - 조회 기간: 2016-02-23 ~ 2016-03-29 (개봉 분포: 2016-03-24 ~ 2016-03-30)
  ✅ 배치 [7] 저장 완료: 144건의 트렌드 레코드 적재

🚀 배치 [8/616] 처리 시작...
  